# Milestone 3 lab, notebook 2 of 8: Which column is the key?

## Where we are

| Notebook | What you do |
|---|---|
| 1. One order, one row | Write the same rows three ways: append, overwrite, upsert |
| **2. Which column is the key?** (you are here) | Merge on the wrong key. Then meet an order that arrives twice in one run |
| 3. Run it twice | Prove that a second run changes nothing |
| 4. At least once, exactly once | Crash the load in three places, and see what each one costs |
| 5. The update that came late | Find a change the watermark has passed. Catch it with a lookback |
| 6. Fix one day | Break Friday's numbers, and rebuild Friday alone |
| 7. The order that vanished | A deleted order, and why the query cannot see it |
| 8. The production version | The three properties, as tests |

An upsert asks "is this order already here?" The **key** answers that. Choose the wrong key, and the upsert quietly destroys data. This notebook does that on purpose, and then meets a second trap.

In [1]:
import psycopg

from lab_helpers import lab_warehouse, land_like_milestone_2, show_doc, table

con = lab_warehouse(fresh=True)
land_like_milestone_2(con, "2026-06-08", "2026-06-09", "2026-06-10", "2026-06-11")

con.execute("""
    CREATE VIEW landed AS
    SELECT landing_id, run_id, page_no,
           (payload->>'order_id')::BIGINT       AS order_id,
           (payload->>'customer_id')::BIGINT    AS customer_id,
           payload->>'status'                   AS status,
           (payload->>'total_cents')::BIGINT    AS total_cents,
           (payload->>'updated_at')::TIMESTAMPTZ AS updated_at
    FROM raw.orders_landing
""")
print("Four mornings are landed.")

Landed 226 rows for the morning of 2026-06-08.
Landed  29 rows for the morning of 2026-06-09.
Landed  38 rows for the morning of 2026-06-10.
Landed  49 rows for the morning of 2026-06-11.
Four mornings are landed.


## Step 1: What a key is

A key is a column that is different for every real thing. For orders, look for a column with as many different values as there are orders.

In [2]:
table(con, """
    SELECT count(DISTINCT order_id) AS different_orders,
           count(DISTINCT customer_id) AS different_customers,
           count(DISTINCT updated_at) AS different_updated_at
    FROM landed WHERE run_id = '2026-06-08_1'
""")

,different_orders,different_customers,different_updated_at
0,226,172,226


226 orders, 226 different `order_id` values. `customer_id` has fewer: some customers ordered more than once.

## Step 2: Merge on the wrong key

Suppose someone writes the merge on `customer_id`. Start with Monday's orders, then merge Tuesday's.

In [3]:
def monday_table(name):
    con.execute(f"DROP TABLE IF EXISTS {name}")
    con.execute(f"CREATE TABLE {name} AS SELECT order_id, customer_id, status, total_cents, updated_at FROM landed WHERE run_id = '2026-06-08_1'")

def merge_tuesday(name, key):
    con.execute(f"""
        MERGE INTO {name} AS t
        USING (SELECT DISTINCT ON ({key}) * FROM landed WHERE run_id = '2026-06-09_1' ORDER BY {key}, updated_at DESC) AS s
           ON t.{key} = s.{key}
        WHEN MATCHED THEN
            UPDATE SET order_id = s.order_id, status = s.status, total_cents = s.total_cents, updated_at = s.updated_at
        WHEN NOT MATCHED THEN
            INSERT (order_id, customer_id, status, total_cents, updated_at)
            VALUES (s.order_id, s.customer_id, s.status, s.total_cents, s.updated_at)
    """)

monday_table("by_customer"); merge_tuesday("by_customer", "customer_id")
monday_table("by_order");    merge_tuesday("by_order", "order_id")

table(con, """
    SELECT 'customer_id (wrong)' AS merged_on, count(*) AS rows, count(DISTINCT order_id) AS different_orders, sum(total_cents) AS total_cents FROM by_customer
    UNION ALL
    SELECT 'order_id (right)', count(*), count(DISTINCT order_id), sum(total_cents) FROM by_order
""")

,merged_on,rows,different_orders,total_cents
0,customer_id (wrong),238,234,581912
1,order_id (right),251,251,609005


No error. Both merges ran. But the table merged on `customer_id` has fewer orders and less money.

Here is what happened. When a customer who ordered last week ordered again on Monday, the merge found "this customer is already here" and **wrote the new order over the old one**. An old order is simply gone.

In [4]:
table(con, """
    SELECT m.order_id AS lost_order, m.customer_id, m.total_cents AS lost_cents
    FROM landed AS m
    WHERE m.run_id = '2026-06-08_1'
      AND m.order_id NOT IN (SELECT order_id FROM by_customer)
    ORDER BY m.order_id
    LIMIT 5
""")

,lost_order,customer_id,lost_cents
0,1003,1276,2992
1,1007,1232,2880
2,1012,1281,2697
3,1024,1207,2249
4,1074,1153,2575


A wrong key does not fail. It overwrites. Always check: **is my key unique for the thing one row means?**

## Step 3: Let the database guard the key

A **primary key** is a promise the database keeps for you: no two rows with the same key. Add it, then try to break it.

In [5]:
con.execute("ALTER TABLE by_order ADD PRIMARY KEY (order_id)")

try:
    con.execute("INSERT INTO by_order SELECT order_id, customer_id, status, total_cents, updated_at FROM landed WHERE order_id = 1130 LIMIT 1")
except psycopg.Error as error:
    print("Postgres refused:")
    print(" ", error.diag.message_primary)

Postgres refused:
  duplicate key value violates unique constraint "by_order_pkey"


Now a mistake becomes an error, and not a silently doubled number.

## Step 4: Three kinds of key

| Word | Meaning | In Arden's warehouse |
|---|---|---|
| **Primary key** | The column the database keeps unique in a table | `order_id` in the clean orders table |
| **Natural key** | A key that comes from the business itself. It means something outside the warehouse | `order_id`: the order system made it, and the customer sees it |
| **Dedupe key** | The key you keep one row per, when the same thing arrived more than once | `order_id` again, in step 5 |

The opposite of a natural key is a key the warehouse invents. `landing_id` is one: 1, 2, 3 … It is unique, but it names a **landed row**, not an order. Order 1130 has two of them.

In [6]:
table(con, "SELECT landing_id, order_id, run_id, status, updated_at FROM landed WHERE order_id = 1130 ORDER BY landing_id")

,landing_id,order_id,run_id,status,updated_at
0,137,1130,2026-06-08_1,delivered,2026-06-05 20:45:00+00:00
1,232,1130,2026-06-09_1,delivered,2026-06-08 16:00:00+00:00


## Step 5: The same order, twice in one run

On Thursday 11 June the order system served one page twice. So Thursday's rows hold some orders two times.

In [7]:
table(con, """
    SELECT count(*) AS rows_landed, count(DISTINCT order_id) AS different_orders
    FROM landed WHERE run_id = '2026-06-11_1'
""")

,rows_landed,different_orders
0,49,37


Merge them into a table that holds the first three mornings.

In [8]:
con.execute("DROP TABLE IF EXISTS orders")
con.execute("""
    CREATE TABLE orders AS
    SELECT DISTINCT ON (order_id) order_id, customer_id, status, total_cents, updated_at
    FROM landed WHERE run_id <> '2026-06-11_1' ORDER BY order_id, updated_at DESC
""")
con.execute("ALTER TABLE orders ADD PRIMARY KEY (order_id)")

MERGE_THURSDAY = """
    MERGE INTO orders AS t
    USING ({source}) AS s ON t.order_id = s.order_id
    WHEN MATCHED THEN UPDATE SET status = s.status, total_cents = s.total_cents, updated_at = s.updated_at
    WHEN NOT MATCHED THEN INSERT VALUES (s.order_id, s.customer_id, s.status, s.total_cents, s.updated_at)
"""

try:
    with con.transaction():
        con.execute(MERGE_THURSDAY.format(source="SELECT * FROM landed WHERE run_id = '2026-06-11_1'"))
except psycopg.Error as error:
    print("Postgres refused:")
    print(" ", error.diag.message_primary)

Postgres refused:
  duplicate key value violates unique constraint "orders_pkey"


The merge stops. It was given the same order twice, and it will not guess which copy to keep. (An append would have taken both without a word.)

So before the merge, keep **one row per key**. When two copies differ, keep the newest. This step is called deduplication, and its key is the dedupe key.

In [9]:
ONE_ROW_PER_ORDER = """
    SELECT DISTINCT ON (order_id) *
    FROM landed
    WHERE run_id = '2026-06-11_1'
    ORDER BY order_id, updated_at DESC, landing_id DESC
"""
table(con, f"SELECT count(*) AS rows, count(DISTINCT order_id) AS different_orders FROM ({ONE_ROW_PER_ORDER}) AS staged")

,rows,different_orders
0,37,37


In [10]:
con.execute(MERGE_THURSDAY.format(source=ONE_ROW_PER_ORDER))
table(con, "SELECT count(*) AS rows, count(DISTINCT order_id) AS different_orders FROM orders")

,rows,different_orders
0,317,317


317 rows, 317 orders. The repeated page did no harm.

`DISTINCT ON (order_id) ... ORDER BY order_id, updated_at DESC` reads: "for each order, keep the first row when sorted newest first". Many teams write the same thing with `ROW_NUMBER()`.

In [11]:
show_doc("LLD.md", "3. Stage: one row per order")

**From `docs/LLD.md`**

> ## 3. Stage: one row per order
> 
> A run can hold the same order more than once: the API can serve a page twice, and the lookback can bring an old version beside a new one. `MERGE` refuses that. So staging keeps the newest version of each order.
> 
> ```sql
> SELECT DISTINCT ON (order_id) *
> FROM   <this run's landed rows, as columns>
> ORDER  BY order_id, updated_at DESC, landing_id DESC;
> ```
> 
> On 11 June the API serves page 2 twice. 49 rows land. Staging keeps 37.

In [12]:
con.close()

The key definitions follow the D8Loop chapters [Loading & Write Patterns](https://www.d8loop.com/modules/knowledge-corner/data-pipeline/loading-and-write-patterns) and [Transformation Fundamentals](https://www.d8loop.com/modules/knowledge-corner/data-pipeline/transformation-fundamentals).

## What you now know

- A merge on the wrong key does not fail. It overwrites other rows.
- A primary key lets the database refuse a duplicate.
- `order_id` is a natural key: it comes from the business. `landing_id` is invented by the warehouse.
- A merge needs one row per key. Deduplicate first, and keep the newest version.

**Say it in an interview:** "Before I merge I check two things about the key. It must be unique for the grain of the target table, or the merge overwrites unrelated rows without an error. And the incoming batch must hold each key once, so I dedupe on the key and keep the latest version, because the API once sent me a page twice."

Next: [notebook 3](03_run_it_twice.ipynb) runs the finished pipeline twice, and proves nothing changes.